# Fourier Series & Fourier Transform
## A hands-on, interactive tour from periodic signals to spectral analysis

The Fourier approach decomposes signals into elementary sinusoidal building blocks.  
Two complementary tools emerge naturally:

| Tool | Domain | Signal class | Spectrum |
|------|--------|-------------|---------|
| **Fourier Series** | Continuous time | Periodic, period $T$ | Discrete lines at $nf_0$ |
| **Fourier Transform** | Continuous time | Aperiodic, finite energy | Continuous function of $f$ |

Both share the same core idea: **represent complexity as a weighted superposition of harmonics.**


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=[
        "#2196F3","#FF5722","#4CAF50","#9C27B0","#FF9800"
    ]),
})
print("All imports OK — let's build some spectra.")


All imports OK — let's build some spectra.


---
## 1 · Fourier Series

A $T$-periodic signal $f(t)$ is exactly represented by

$$f(t) = \frac{a_0}{2} + \sum_{n=1}^{\infty}\!\Bigl[a_n\cos(n\omega_0 t) + b_n\sin(n\omega_0 t)\Bigr], \qquad \omega_0 = \frac{2\pi}{T}$$

The **synthesis coefficients** are projections of $f$ onto orthogonal sinusoids:

$$a_n = \frac{2}{T}\int_0^T f(t)\cos(n\omega_0 t)\,dt, \qquad b_n = \frac{2}{T}\int_0^T f(t)\sin(n\omega_0 t)\,dt$$

Closed-form results for three canonical odd signals (only sine terms survive by symmetry):

| Signal | $b_n$ (odd $n$ only) |
|--------|---------------------|
| Square | $\dfrac{4}{n\pi}$ |
| Sawtooth | $\dfrac{2(-1)^{n+1}}{n\pi}$ (all $n$) |
| Triangle | $\dfrac{8(-1)^{(n-1)/2}}{n^2\pi^2}$ |

Adding more harmonics sharpens the approximation; **the amplitude spectrum** — $|b_n|$ vs $n$ — reveals how energy is distributed across harmonics.

> **Slider guide** — drag *Harmonics N* to watch successive terms sharpen the waveform, enable *Show components* to see individual sinusoids stacking up.


In [ ]:
def _fs_approx(kind, N, t):
    """Return (approximation, list-of-components, harmonic-indices, coefficients)."""
    y, comps, ns, cs = np.zeros_like(t), [], [], []
    if kind == "Square":
        for k in range(N):
            n = 2*k + 1
            c = 4 / (np.pi * n)
            comp = c * np.sin(n * t)
            y += comp; comps.append(comp); ns.append(n); cs.append(abs(c))
    elif kind == "Sawtooth":
        for k in range(1, N+1):
            c = 2 * (-1)**(k+1) / (np.pi * k)
            comp = c * np.sin(k * t)
            y += comp; comps.append(comp); ns.append(k); cs.append(abs(c))
    else:  # Triangle
        for k in range(N):
            n = 2*k + 1
            c = 8 * (-1)**k / (np.pi**2 * n**2)
            comp = c * np.sin(n * t)
            y += comp; comps.append(comp); ns.append(n); cs.append(abs(c))
    return y, comps, ns, cs

def _target(kind, t):
    T = {"Square": np.sign(np.sin(t)),
         "Sawtooth": t / np.pi,
         "Triangle": 2*np.abs(t/np.pi) - 1}
    return T[kind]

def plot_series(N=5, Signal="Square", Show_components=False):
    t = np.linspace(-np.pi, np.pi, 2000)
    approx, comps, ns, cs = _fs_approx(Signal, N, t)
    target = _target(Signal, t)
    rms = np.sqrt(np.mean((target - approx)**2))

    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[2.2, 1], wspace=0.35)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    if Show_components and N <= 12:
        for comp in comps:
            ax1.plot(t, comp, lw=0.9, alpha=0.35)
    ax1.plot(t, target, "k--", lw=1.8, alpha=0.6, label="Target")
    ax1.plot(t, approx, lw=2.2, label=f"N = {N} harmonics")
    ax1.set_xlim(-np.pi, np.pi)
    ax1.set_ylim(-1.6, 1.6)
    ax1.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax1.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
    ax1.set_xlabel("t"); ax1.set_ylabel("Amplitude")
    ax1.set_title(f"{Signal} wave — Fourier series  (RMS error = {rms:.4f})")
    ax1.legend(loc="upper right", fontsize=9)

    markerline, stemlines, baseline = ax2.stem(ns, cs, linefmt="C0-",
                                                markerfmt="C0o", basefmt="k-")
    plt.setp(stemlines, lw=1.6); plt.setp(markerline, ms=6)
    ax2.set_xlabel("Harmonic n"); ax2.set_ylabel("|coefficient|")
    ax2.set_title("Amplitude spectrum")
    plt.show()

interact(
    plot_series,
    N=widgets.IntSlider(value=5, min=1, max=50, step=1,
                        description="Harmonics N", style={"description_width":"110px"},
                        layout=widgets.Layout(width="420px")),
    Signal=widgets.Dropdown(options=["Square","Sawtooth","Triangle"],
                             description="Signal",
                             style={"description_width":"110px"}),
    Show_components=widgets.Checkbox(value=False, description="Show components"),
)


interactive(children=(IntSlider(value=5, description='Harmonics N', layout=Layout(width='420px'), max=50, min=…

<function __main__.plot_series(N=5, Signal='Square', Show_components=False)>

---
## 2 · The Gibbs Phenomenon

At a jump discontinuity, the partial sum always **overshoots by ≈ 9 %** of the jump height, regardless of how many harmonics are added. The overshoot narrows but never disappears — it is not a numerical artefact but a genuine property of pointwise convergence of Fourier series.

$$\lim_{N\to\infty}\max_t\,|f_N(t) - f(t)| \approx 0.0895 \cdot \Delta$$

where $\Delta$ is the jump size. Convergence is **uniform** only away from discontinuities.

> **Slider** — push *N* to 100 and observe that the overshoot spike narrows but stays near 9 % in the zoomed panel.


In [ ]:
def plot_gibbs(N=5):
    t = np.linspace(-np.pi, np.pi, 8000)
    approx = sum((4/(np.pi*(2*k+1)))*np.sin((2*k+1)*t) for k in range(N))
    target = np.sign(np.sin(t))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(t, target, "k--", lw=1.5, alpha=0.55, label="Square wave")
    ax1.plot(t, approx, lw=2,   label=f"N = {N}")
    ax1.set_xlim(-np.pi, np.pi); ax1.set_ylim(-1.5, 1.5)
    ax1.set_xlabel("t"); ax1.set_title("Full signal")
    ax1.legend()

    # Zoom around first positive transition (t ~ 0)
    mask = (t > -0.8) & (t < 0.8)
    peak = approx[mask].max()
    overshoot_pct = (peak - 1.0) * 100
    ax2.plot(t[mask], target[mask], "k--", lw=1.5, alpha=0.55)
    ax2.plot(t[mask], approx[mask], lw=2)
    ax2.axhline(peak, color="C1", ls=":", lw=1.5,
                label=f"Peak ≈ {peak:.3f}  (overshoot {overshoot_pct:+.1f} %)")
    ax2.axhline(1.0, color="k",  ls="-",  lw=0.8, alpha=0.4)
    ax2.set_xlabel("t"); ax2.set_title("Zoomed near discontinuity")
    ax2.legend(fontsize=9)
    plt.tight_layout(); plt.show()

interact(
    plot_gibbs,
    N=widgets.IntSlider(value=5, min=1, max=100, step=1,
                        description="Harmonics N",
                        style={"description_width":"110px"},
                        layout=widgets.Layout(width="420px")),
)


interactive(children=(IntSlider(value=5, description='Harmonics N', layout=Layout(width='420px'), min=1, style…

<function __main__.plot_gibbs(N=5)>

---
## 3 · The Fourier Transform

For an aperiodic, finite-energy signal the period $T\to\infty$ and the discrete line spectrum becomes a **continuous** function:

$$X(f) = \int_{-\infty}^{\infty} x(t)\,e^{-j2\pi f t}\,dt \qquad \longleftrightarrow \qquad x(t) = \int_{-\infty}^{\infty} X(f)\,e^{+j2\pi f t}\,df$$

**Parseval's theorem** states that energy is conserved in both domains:
$$\int_{-\infty}^{\infty}|x(t)|^2\,dt = \int_{-\infty}^{\infty}|X(f)|^2\,df$$

Key analytic pairs used below:

| $x(t)$ | $X(f)$ |
|--------|--------|
| $\Pi(t/\tau)$ — rectangular pulse, width $\tau$ | $\tau\,\text{sinc}(f\tau)$ |
| $e^{-t^2/(2\sigma^2)}$ — Gaussian | $\sigma\sqrt{2\pi}\,e^{-2\pi^2\sigma^2 f^2}$ |
| $\Lambda(t/\tau)$ — triangle, base $2\tau$ | $\tau\,\text{sinc}^2(f\tau)$ |

> **Critical insight** — wider pulses produce *narrower* spectra. This is the **time-bandwidth duality**: stretching a signal in time compresses its spectrum by the same factor.

> **Slider guide** — adjust *Width* and watch the spectrum breathe in the opposite direction.


In [4]:
def _make_pulse(kind, width, t):
    hw = width / 2
    if kind == "Rectangular":
        return np.where(np.abs(t) <= hw, 1.0, 0.0)
    elif kind == "Gaussian":
        sigma = width / 5          # 5-sigma span ≈ width
        return np.exp(-t**2 / (2*sigma**2))
    else:  # Triangle
        return np.maximum(0.0, 1 - np.abs(t) / hw)

def _analytical_X(kind, width, f):
    hw = width / 2
    if kind == "Rectangular":
        return width * np.sinc(f * width)
    elif kind == "Gaussian":
        sigma = width / 5
        return sigma * np.sqrt(2*np.pi) * np.exp(-2*np.pi**2*sigma**2*f**2)
    else:  # Triangle
        return hw * np.sinc(f * hw)**2

def plot_ft(Signal="Rectangular", Width=1.0):
    t = np.linspace(-6, 6, 6000)
    f = np.linspace(-6, 6, 6000)
    x = _make_pulse(Signal, Width, t)
    X = _analytical_X(Signal, Width, f)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(t, x, lw=2.2)
    axes[0].fill_between(t, x, alpha=0.18)
    axes[0].set_xlim(-5, 5); axes[0].set_ylim(-0.15, 1.25)
    axes[0].set_xlabel("t (s)"); axes[0].set_ylabel("x(t)")
    axes[0].set_title(f"{Signal} pulse  —  width = {Width:.2f} s")

    Xabs = np.abs(X)
    axes[1].plot(f, Xabs, "C1", lw=2.2)
    axes[1].fill_between(f, Xabs, alpha=0.18, color="C1")
    axes[1].set_xlim(-5, 5); axes[1].set_ylim(bottom=-0.02)
    axes[1].set_xlabel("f (Hz)"); axes[1].set_ylabel("|X(f)|")
    axes[1].set_title("Magnitude spectrum  |X(f)|")

    # −3 dB bandwidth annotation
    peak = Xabs.max()
    half = peak / np.sqrt(2)
    above = Xabs >= half
    if above.any():
        bw = f[above][-1] - f[above][0]
        axes[1].axhline(half, color="C3", ls="--", lw=1.4, label=f"−3 dB  BW ≈ {bw:.2f} Hz")
        axes[1].legend(fontsize=9)

    # Parseval check
    dt = t[1]-t[0]; df = f[1]-f[0]
    E_t = np.trapezoid(x**2, t)
    E_f = np.trapezoid(Xabs**2, f)
    fig.suptitle(f"Parseval check  —  E_t = {E_t:.3f},  E_f = {E_f:.3f}", fontsize=9, color="gray")
    plt.tight_layout(); plt.show()

interact(
    plot_ft,
    Signal=widgets.Dropdown(options=["Rectangular","Gaussian","Triangle"],
                             description="Signal",
                             style={"description_width":"80px"}),
    Width=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1,
                              description="Width (s)",
                              style={"description_width":"80px"},
                              layout=widgets.Layout(width="400px")),
)


interactive(children=(Dropdown(description='Signal', options=('Rectangular', 'Gaussian', 'Triangle'), style=De…

<function __main__.plot_ft(Signal='Rectangular', Width=1.0)>

---
## 4 · DFT, FFT and Spectral Leakage

The **Discrete Fourier Transform** is the practical gateway to spectral analysis:

$$X[k] = \sum_{n=0}^{N-1} x[n]\,e^{-j2\pi kn/N}, \qquad k = 0,\dots,N-1$$

The FFT computes this in $\mathcal{O}(N\log N)$ instead of $\mathcal{O}(N^2)$.

**Spectral leakage** occurs when the signal frequency is *not* an exact integer multiple of $f_0 = f_s/N$.  
Because the DFT implicitly assumes the $N$-sample block repeats periodically, a non-integer frequency creates a discontinuity at the block boundary → energy spreads to neighbouring bins.

**Windowing** tapers the signal to zero at both ends, smoothing the artificial edge and dramatically reducing side-lobe energy — at the cost of slightly wider main lobes.

| Window | Main-lobe width | Peak side-lobe |
|--------|----------------|----------------|
| Rectangular (none) | $2/N$ | −13 dB |
| Hann | $4/N$ | −31 dB |
| Hamming | $4/N$ | −42 dB |
| Blackman | $6/N$ | −58 dB |

> **Slider guide** — set a non-integer frequency (e.g. 5.3 Hz) and toggle windows to see leakage suppression in the dB plot.


In [5]:
def plot_dft(Frequency=5.3, N_samples=128, Window="Rectangular"):
    fs = N_samples          # 1 second of signal at fs Hz → N samples
    t  = np.arange(N_samples) / fs
    x  = np.sin(2*np.pi*Frequency*t)

    wins = {
        "Rectangular": np.ones(N_samples),
        "Hann":        np.hanning(N_samples),
        "Hamming":     np.hamming(N_samples),
        "Blackman":    np.blackman(N_samples),
    }
    w     = wins[Window]
    x_win = x * w
    freqs = np.fft.rfftfreq(N_samples, 1/fs)
    X_raw = np.fft.rfft(x)      / N_samples
    X_win = np.fft.rfft(x_win)  / N_samples

    fig, axes = plt.subplots(2, 2, figsize=(14, 7))

    axes[0,0].plot(t, x,     lw=1.5, label="Signal x(t)")
    axes[0,0].plot(t, w,     lw=1.2, ls="--", alpha=0.85, label=f"{Window} window")
    axes[0,0].set_xlabel("t (s)"); axes[0,0].set_title("Signal & window function")
    axes[0,0].legend(fontsize=9)

    axes[0,1].plot(t, x_win, lw=1.5, color="C2")
    axes[0,1].set_xlabel("t (s)"); axes[0,1].set_title("Windowed signal")

    for ax, X, label in [
        (axes[1,0], X_raw, "No window"),
        (axes[1,0], X_win, Window),
    ]:
        ax.plot(freqs, np.abs(X), lw=1.6, label=label)
    axes[1,0].axvline(Frequency, color="C3", ls="--", lw=1.2, label=f"True f = {Frequency} Hz")
    axes[1,0].set_xlim(0, fs//2); axes[1,0].set_xlabel("f (Hz)")
    axes[1,0].set_ylabel("Magnitude"); axes[1,0].set_title("Magnitude spectrum")
    axes[1,0].legend(fontsize=9)

    eps = 1e-12
    for X, label in [(X_raw, "No window"), (X_win, Window)]:
        axes[1,1].plot(freqs, 20*np.log10(np.abs(X)+eps), lw=1.6, label=label)
    axes[1,1].axvline(Frequency, color="C3", ls="--", lw=1.2, label=f"True f = {Frequency} Hz")
    axes[1,1].set_xlim(0, min(fs//2, 40)); axes[1,1].set_ylim(-90, 5)
    axes[1,1].set_xlabel("f (Hz)"); axes[1,1].set_ylabel("dB")
    axes[1,1].set_title("Log spectrum — leakage visible here")
    axes[1,1].legend(fontsize=9)

    plt.suptitle(f"DFT  —  f = {Frequency} Hz, N = {N_samples} samples, fs = {fs} Hz", fontsize=11)
    plt.tight_layout(); plt.show()

interact(
    plot_dft,
    Frequency=widgets.FloatSlider(value=5.3, min=1.0, max=25.0, step=0.1,
                                   description="Frequency (Hz)",
                                   style={"description_width":"130px"},
                                   layout=widgets.Layout(width="460px")),
    N_samples=widgets.Dropdown(options=[32,64,128,256,512], value=128,
                                description="N samples",
                                style={"description_width":"130px"}),
    Window=widgets.Dropdown(options=["Rectangular","Hann","Hamming","Blackman"],
                             description="Window",
                             style={"description_width":"130px"}),
)


interactive(children=(FloatSlider(value=5.3, description='Frequency (Hz)', layout=Layout(width='460px'), max=2…

<function __main__.plot_dft(Frequency=5.3, N_samples=128, Window='Rectangular')>

---
## 5 · Time-Frequency Uncertainty

No signal can be simultaneously concentrated in both time and frequency. The fundamental bound is:

$$\sigma_t \cdot \sigma_f \;\geq\; \frac{1}{4\pi}$$

where $\sigma_t$ and $\sigma_f$ are the RMS spreads (standard deviations) of $|x(t)|^2$ and $|X(f)|^2$.

The **Gaussian pulse** is the unique signal that achieves the equality — it is its own Fourier transform (up to scaling). For $x(t) = e^{-t^2/(2\sigma_t^2)}$:

$$X(f) = \sigma_t\sqrt{2\pi}\; e^{-2\pi^2\sigma_t^2 f^2} \implies \sigma_f = \frac{1}{2\pi\sigma_t} \implies \sigma_t\sigma_f = \frac{1}{2\pi} \cdot \frac{1}{2} = \frac{1}{4\pi}$$

This is the signal-processing analogue of Heisenberg's uncertainty principle. It has direct implications for:

- **Radar / sonar** — a short pulse gives good range resolution but poor Doppler resolution.
- **Audio** — sharp transients sound "click-like" and occupy broad bandwidth.
- **Communications** — sharper pulses require more bandwidth; there is no free lunch.

> **Slider** — increase $\sigma_t$ (wider pulse) and watch $\sigma_f$ shrink in lock-step; the product (title) stays pinned at $1/4\pi$.


In [6]:
def plot_uncertainty(sigma_t=1.0):
    t = np.linspace(-12, 12, 8000)
    f = np.linspace(-4,   4, 8000)

    x    = np.exp(-t**2 / (2*sigma_t**2))
    sigma_f = 1 / (2*np.pi*sigma_t)
    Xabs = sigma_t * np.sqrt(2*np.pi) * np.exp(-2*np.pi**2*sigma_t**2*f**2)

    # Numerical verification of sigma_t and sigma_f
    x2  = x**2;  x2 /= np.trapezoid(x2, t)
    mu_t  = np.trapezoid(t   * x2, t)
    s_t   = np.sqrt(np.trapezoid((t-mu_t)**2 * x2, t))
    Xabs2 = Xabs**2; Xabs2 /= np.trapezoid(Xabs2, f)
    mu_f  = np.trapezoid(f   * Xabs2, f)
    s_f   = np.sqrt(np.trapezoid((f-mu_f)**2 * Xabs2, f))
    product = s_t * s_f
    limit   = 1 / (4*np.pi)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
    span = max(5*sigma_t, 2.0)

    ax1.plot(t, x, lw=2.2)
    ax1.axvspan(-sigma_t, sigma_t, alpha=0.18, color="C0",
                label=f"±σₜ = {sigma_t:.2f} s")
    ax1.set_xlim(-span, span); ax1.set_ylim(-0.05, 1.15)
    ax1.set_xlabel("t (s)"); ax1.set_ylabel("x(t)")
    ax1.set_title("Gaussian pulse — time domain")
    ax1.legend(fontsize=9)

    ax2.plot(f, Xabs, "C1", lw=2.2)
    ax2.axvspan(-sigma_f, sigma_f, alpha=0.18, color="C1",
                label=f"±σ_f = {sigma_f:.3f} Hz")
    ax2.set_xlim(-4, 4); ax2.set_ylim(-0.02, Xabs.max()*1.15)
    ax2.set_xlabel("f (Hz)"); ax2.set_ylabel("|X(f)|")
    ax2.set_title("Magnitude spectrum — frequency domain")
    ax2.legend(fontsize=9)

    color = "darkgreen" if abs(product-limit)/limit < 0.01 else "sienna"
    fig.suptitle(
        rf"σₜ · σ_f = {product:.5f}   (minimum = 1/4π ≈ {limit:.5f})",
        fontsize=11, color=color, y=1.01
    )
    plt.tight_layout(); plt.show()

interact(
    plot_uncertainty,
    sigma_t=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05,
                                 description="σₜ (s)",
                                 style={"description_width":"80px"},
                                 layout=widgets.Layout(width="400px")),
)


interactive(children=(FloatSlider(value=1.0, description='σₜ (s)', layout=Layout(width='400px'), max=3.0, min=…

<function __main__.plot_uncertainty(sigma_t=1.0)>